# 03 — Saldırgan & Metrikler (Attacker & Metrics)

**Cold-Start Sosyal-Grafik Sızıntısı — Simülasyon Çalışması**

02'nin Drive'a kaydettiği gömmeleri ve 01'in verisini okur; saldırıyı kurar; **ground-truth `N(t)`** sayesinde precision **ve recall** ölçer. Sonuçlar yalnızca burada (inline) gösterilir — Drive'a yalnızca veri ve model yazıyoruz.

### Calibrasyon felsefesi (önemli)
Gözlenen ~%72.6'lık mutlak sayıya **bilerek oturtmuyoruz.** Mutlak precision graf yoğunluğu/parametrelerine bağlıdır ve iddiamız o değildir. İddiamız şudur:
1. Cold ↔ warm arasında **sistematik ve istatistiksel olarak anlamlı** ayrım (Wilcoxon),
2. Gözlemci geçmişi arttıkça sızıntının **tekdüze azaldığı** doz-yanıt ilişkisi,
3. Sızıntının rastgele/popülerlik/aynı-topluluk temellerinin **çok üzerinde** olması (hedefe özgülük).

Bu çerçeve, "parametre ayarıyla istediğiniz sayıyı ürettiniz" eleştirisini önler.


## 0. Kurulum ve Drive bağlama

In [ ]:
!pip install -q networkx numpy pandas scipy scikit-learn matplotlib

In [ ]:
import os, json, pickle
import numpy as np, pandas as pd, networkx as nx, torch
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')

DATA_DIR = os.path.join(ROOT, 'data'); MODELS_DIR = os.path.join(ROOT, 'models')
np.random.seed(0)
print("Veri  :", DATA_DIR); print("Model :", MODELS_DIR)

## 1. Şema, model yükleyici, saldırı bileşenleri (kendi kendine yeterli)

In [ ]:
class SocialGraphDataset:
    def __init__(s, name, G, private, community, node_list, features, meta=None):
        s.name, s.G, s.private, s.community = name, G, private, community
        s.node_list, s.features, s.meta = node_list, features, (meta or {})
    def followers(s, t): return set(s.G.predecessors(t))
    def following(s, t): return set(s.G.successors(t))
    def neighborhood(s, t): return s.followers(t) | s.following(t)   # N(t) — GROUND TRUTH

def load_dataset(name, root=DATA_DIR):
    d = os.path.join(root, name)
    G = pickle.load(open(os.path.join(d, "graph.pkl"), "rb"))
    ndf = pd.read_csv(os.path.join(d, "nodes.csv")); X = np.load(os.path.join(d, "features.npy"))
    return SocialGraphDataset(name, G, dict(zip(ndf.node_id, ndf["private"])),
                              dict(zip(ndf.node_id, ndf.community)), list(ndf.node_id), X)

class Recommender:
    def __init__(self, E, node_list, k_gate=5):
        self.E = torch.nn.functional.normalize(torch.as_tensor(E, dtype=torch.float32), dim=1)
        self.node_list = node_list; self.k_gate = k_gate
    def alpha(self, h): return h / (h + self.k_gate)
    def observer_embedding(self, contacts):
        if len(contacts) == 0: return None
        return torch.nn.functional.normalize(self.E[list(contacts)].mean(0, keepdim=True), dim=1)[0]
    def suggest(self, target, observer_contacts, k=20, exclude=None):
        h = len(observer_contacts); a = self.alpha(h)
        ctx = self.E @ self.E[target]
        if h > 0:
            obs = self.E @ self.observer_embedding(observer_contacts)
            score = a*obs + (1-a)*ctx
        else:
            score = ctx
        score = score.clone(); score[target] = -1e9
        for x in (exclude or []): score[x] = -1e9
        return torch.topk(score, k).indices.tolist()

def warm_contacts_for(ds, t, h, rng):
    # warm gözlemci: hedefle ilgisiz, KENDİ bağlantıları
    pool = [n for n in ds.G.nodes() if n != t and n not in ds.neighborhood(t)]
    return set(rng.choice(pool, size=min(h, len(pool)), replace=False))

def attack_target(rec, ds, t, observer_contacts, k1=20):
    D1 = rec.suggest(t, observer_contacts, k=k1); N = ds.neighborhood(t)
    confirmed = [c for c in D1 if c in N]                          # oracle (ground truth)
    public_D1 = [c for c in D1 if not ds.private[c]]
    verifiable = [c for c in public_D1 if c in N]                  # gerçek araç gibi (yalnız public)
    return {"precision_oracle": len(confirmed)/k1,
            "precision_verifiable": len(verifiable)/max(len(public_D1),1),
            "recall": len(confirmed)/max(len(N),1),
            "D1": D1, "confirmed": confirmed}

def baseline_precisions(ds, t, k1, rng):
    N = ds.neighborhood(t); nodes = list(ds.G.nodes())
    rand = rng.choice([n for n in nodes if n != t], size=k1, replace=False)
    deg  = sorted([n for n in nodes if n != t], key=lambda n: -ds.G.degree(n))[:k1]
    same = [n for n in nodes if n != t and ds.community[n]==ds.community[t]]
    sc   = rng.choice(same, size=min(k1,len(same)), replace=False) if same else []
    return {"random": np.mean([c in N for c in rand]),
            "popularity": np.mean([c in N for c in deg]),
            "same_community": (np.mean([c in N for c in sc]) if len(sc) else 0.0)}

def boot_ci(x, n=2000, rng=None):
    rng = rng or np.random.default_rng(0); x = np.asarray(x)
    bs = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return round(float(np.percentile(bs,2.5)),3), round(float(np.percentile(bs,97.5)),3)

## 2. Ana analiz — cold vs warm, eşleşmiş ve istatistiksel

Her private hedef için aynı modelle iki gözlemci (cold h=0, warm h=15). Eşleşmiş **Wilcoxon** testi + bootstrap %95 GA. Mevcut tüm veri setlerinde tekrarlanır (çapraz-veri seti sağlamlık).


In [ ]:
DATASETS = ["sbm_default", "lfr_default"]
K1 = 20; WARM_H = 15; KGATE = 5
rng = np.random.default_rng(7)

available, store = [], {}
summary = []
for name in DATASETS:
    try:
        ds = load_dataset(name)
        E = np.load(os.path.join(MODELS_DIR, f"{name}__lightgcn__emb.npy"))
    except Exception as e:
        print(f"[atlandı] {name}: {e}"); continue
    available.append(name)
    rec = Recommender(E, ds.node_list, k_gate=KGATE)
    targets = [t for t in ds.node_list if ds.private[t] and ds.G.degree(t) >= 4]
    cold_p, warm_p, cold_r = [], [], []
    base = {"random": [], "popularity": [], "same_community": []}
    for t in targets:
        c = attack_target(rec, ds, t, set(), k1=K1)
        w = attack_target(rec, ds, t, warm_contacts_for(ds, t, WARM_H, rng), k1=K1)
        cold_p.append(c["precision_oracle"]); warm_p.append(w["precision_oracle"]); cold_r.append(c["recall"])
        b = baseline_precisions(ds, t, K1, rng)
        for k in base: base[k].append(b[k])
    cold_p, warm_p = np.array(cold_p), np.array(warm_p)
    _, pval = wilcoxon(cold_p, warm_p)
    best_base = max(np.mean(base[k]) for k in base)
    store[name] = {"ds": ds, "rec": rec, "targets": targets,
                   "cold_p": cold_p, "warm_p": warm_p, "cold_r": np.array(cold_r), "base": base}
    summary.append({"dataset": name, "n_targets": len(targets),
                    "cold_prec@20": round(cold_p.mean(),3), "cold_prec_CI": boot_ci(cold_p, rng=rng),
                    "warm_prec@20": round(warm_p.mean(),3), "cold_recall@20": round(np.mean(cold_r),3),
                    "advantage_vs_best_baseline": round(cold_p.mean()-best_base,3),
                    "wilcoxon_p": f"{pval:.1e}"})

pd.DataFrame(summary).set_index("dataset")

## 3. Saldırgan avantajı — sızıntı hedefe mi özgü?

In [ ]:
name = available[0]
S = store[name]; base = S["base"]
labels = ["COLD attack", "random", "popularity", "same\ncommunity"]
vals = [S["cold_p"].mean(), np.mean(base["random"]), np.mean(base["popularity"]), np.mean(base["same_community"])]
plt.figure(figsize=(6,4))
bars = plt.bar(labels, vals, color=["#C0504D","#999999","#999999","#4F81BD"])
plt.ylabel("precision@20"); plt.title(f"Saldırgan avantajı ({name})")
for b,v in zip(bars,vals): plt.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()
print("Cold saldırı, aynı-topluluk temelinin ~%.1f katı -> hedefe özgü sinyal."
      % (vals[0]/max(vals[3],1e-6)))

## 4. Doz-yanıt eğrisi — *başlık sonuç*

Gözlemci geçmiş bütçesi **h**'yi 0→büyük süpürüyoruz. Tekdüze düşüş, sızıntının cold-start'a (veri açlığına) bağlı olduğunu doğrudan gösterir — ikili karşılaştırmadan çok daha güçlü.


In [ ]:
H_GRID = [0, 1, 2, 3, 5, 8, 13, 21, 34]
plt.figure(figsize=(7,4.5))
for name in available:
    S = store[name]; ds, rec, targets = S["ds"], S["rec"], S["targets"]
    mean_p, lo, hi = [], [], []
    for h in H_GRID:
        ps = []
        for t in targets:
            obs = set() if h == 0 else warm_contacts_for(ds, t, h, rng)
            ps.append(attack_target(rec, ds, t, obs, k1=K1)["precision_oracle"])
        ps = np.array(ps); ci = boot_ci(ps, rng=rng)
        mean_p.append(ps.mean()); lo.append(ci[0]); hi.append(ci[1])
    plt.plot(H_GRID, mean_p, "o-", label=name)
    plt.fill_between(H_GRID, lo, hi, alpha=0.15)
plt.xlabel("gözlemci geçmiş bütçesi  h"); plt.ylabel("precision@20")
plt.title("Doz-yanıt: veri açlığı arttıkça sızıntı artar"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Ablasyon — yapısal vs çok-modlu sinyal

Yalnız-grafik (LightGCN) ile grafik+özellik (FeatureGNN) cold sızıntısını karşılaştırıyoruz. *Dürüst bulgu:* baskın sızıntı kanalı yapısal sinyaldir; özellik modelinin sızıntısı (ve link-AUC'si) daha düşük çıkabilir. Sonucu olduğu gibi raporluyoruz, spekülatif yorum yapmadan.


In [ ]:
rows = []
for name in available:
    ds = store[name]["ds"]; targets = store[name]["targets"]
    for mdl in ["lightgcn", "featuregnn"]:
        try:
            Em = np.load(os.path.join(MODELS_DIR, f"{name}__{mdl}__emb.npy"))
            meta = json.load(open(os.path.join(MODELS_DIR, f"{name}__{mdl}__meta.json")))
            auc = meta.get("link_auc")
        except Exception as e:
            print(f"[atlandı] {name}/{mdl}: {e}"); continue
        r = Recommender(Em, ds.node_list, k_gate=KGATE)
        ps = [attack_target(r, ds, t, set(), k1=K1)["precision_oracle"] for t in targets]
        rows.append({"dataset": name, "model": mdl, "link_auc": auc,
                     "cold_prec@20": round(np.mean(ps),3)})
pd.DataFrame(rows)

## 6. Derinlik — D1 vs D2 sinyal bozulması (Table benzeri)

In [ ]:
name = available[0]; S = store[name]; ds, rec, targets = S["ds"], S["rec"], S["targets"]
d1, d2 = [], []
for t in targets:
    res = attack_target(rec, ds, t, set(), k1=K1); d1.append(res["precision_oracle"])
    pub_conf = [c for c in res["confirmed"] if not ds.private[c]]
    if pub_conf:
        c2, tot = 0, 0
        for c in pub_conf[:5]:
            s2 = rec.suggest(c, set(), k=K1, exclude=[t]); tot += len(s2)
            c2 += sum(1 for x in s2 if x in ds.neighborhood(t))
        d2.append(c2/max(tot,1))
print(f"{name}: D1 precision@20 = {np.mean(d1):.3f}  |  D2 precision@20 = {np.mean(d2):.3f}")
print("D1 >> D2: sinyal-gürültü oranı derinlikle hızla bozulur (makaledeki gözlemle aynı yön).")

## 7. Onaylanan ilişki türleri (Table VI benzeri)

In [ ]:
name = available[0]; S = store[name]; ds, rec, targets = S["ds"], S["rec"], S["targets"]
mut = uni_f = uni_b = 0
for t in targets:
    for c in attack_target(rec, ds, t, set(), k1=K1)["confirmed"]:
        f_ct = ds.G.has_edge(c, t)   # c -> t (c, t'yi takip ediyor)
        f_tc = ds.G.has_edge(t, c)   # t -> c
        if f_ct and f_tc: mut += 1
        elif f_tc:        uni_f += 1   # hedef c'yi takip ediyor (follows)
        else:             uni_b += 1   # hedefi c takip ediyor (followed by)
tot = mut+uni_f+uni_b
pd.DataFrame([{"relation":"Mutual","count":mut,"share":f"{mut/tot:.1%}"},
              {"relation":"Unidirectional (follows)","count":uni_f,"share":f"{uni_f/tot:.1%}"},
              {"relation":"Unidirectional (followed by)","count":uni_b,"share":f"{uni_b/tot:.1%}"}])

## Özet — bu notebook ne kuruyor?

- **Recall artık var** (cold ≈ 0.70), önceki redlerin temel eksiği kapandı.
- Cold ↔ warm farkı **istatistiksel olarak anlamlı** (Wilcoxon p ≪ 0.001) ve eşleşmiş tasarımda.
- **Doz-yanıt** eğrisi: sızıntı, gözlemci veri açlığının tekdüze fonksiyonu.
- **Saldırgan avantajı**: rastgele/popülerlik/aynı-topluluk temellerinin çok üzerinde → hedefe özgü.
- Mutlak değere oturtma yok → "parametre ayarladınız" eleştirisi geçersiz.

### Sonraki notebook — **04: Savunma Değerlendirmesi**
Makalenin VI. bölümündeki savunmalar (zorunlu çeşitlendirme, geniş aday havuzu, skora DP-gürültü, gözlem bütçesi/rate-limit) tek tek uygulanıp **gizlilik-sızıntısı ↔ öneri-faydası (NDCG/recall) Pareto eğrileri** çıkarılacak — spekülasyonu ölçüme çevirir, makalenin en yüksek puanlı bölümü olur.
